# Feature-representation ablation (paper Sect. 2.4/3.1), seeded and cross-validated

The draft's Sect. 3.1 numbers come from single unseeded runs. The learned-embedding result was quoted from a
separate script rather than run in the notebook. Here all three representations are trained with the same
code on the 243-molecule CHO pool:

- **A, raw counts**: the subgroup-count vector (34 dimensions).
- **B, hand-crafted**: AIOMFAC main-group fractions, O:C, N:C, ln M and ln(1 + q).
- **C, learned embedding**: a trainable 16-dimensional vector per subgroup, sum-pooled by count (Deep-Sets
  style), concatenated with (w, T) and trained end to end with the same head. Sum-pooling learned embeddings
  by count is exactly a bias-free linear layer applied to the count vector, which is how it is implemented.

Evaluation is in two parts: (1) the paper's hash split (33 held-out molecules) over 5 seeds; (2) five-fold
molecule-level cross-validation (folds by SMILES hash) over 3 seeds, so that every molecule is held out once.
Training follows Part I: Adam with lr 1e-3 and wd 1e-5, batch 256, early stopping with patience 30 on 10 % of
the training points, at most 500 epochs, and 20 w × 4 T labels per molecule.

In [1]:
import json, time, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
from rdkit import Chem, RDLogger
from aiomfac_py import ActivityModel, Component
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py.params import load_subgroup_params
import aiomfac_py.s2as._mapping as _m
_m.print = lambda *a, **k: None; RDLogger.DisableLog("rdApp.*")
sg = load_subgroup_params(); WATER = Component(1, "Water", ((16, 1),))

pool = []
for d in json.load(open("bimog_unified.json")):
    if d["#N"] != 0 or d["#Hal"] != 0:
        continue
    r = smiles_to_components([d["smiles"]], water_as_component1=True)
    mh = Chem.AddHs(Chem.MolFromSmiles(d["smiles"])); sym = [a.GetSymbol() for a in mh.GetAtoms()]; nc = max(sym.count("C"), 1)
    pool.append({"name": d["name"], "smiles": d["smiles"], "subgroups": r.components[1].subgroups,
                 "M": sum(a.GetMass() for a in mh.GetAtoms()), "oc": sym.count("O") / nc, "nc": sym.count("N") / nc})
print(len(pool), "CHO molecules")

rng = np.random.RandomState(0); rows = []
for oi, o in enumerate(pool):
    m = ActivityModel([WATER, Component(2, "o", tuple(o["subgroups"]))]); e = np.linspace(0.02, 0.98, 21)
    for w in rng.uniform(e[:-1], e[1:]):
        for T in rng.uniform(273.15, 313.15, 4):
            lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
            if np.isfinite(lg[:2]).all(): rows.append((oi, w, T, lg[0], lg[1]))
D = np.array(rows); PM = D[:, 0].astype(int)
CT = np.column_stack([D[:, 1], (D[:, 2] - 293.15) / 20]).astype(np.float32); Y = D[:, 3:5].astype(np.float32)

vocab = sorted({int(s) for o in pool for s, q in o["subgroups"]})
COUNTS = np.zeros((len(pool), len(vocab)), np.float32)
for i, o in enumerate(pool):
    for s, q in o["subgroups"]:
        COUNTS[i, vocab.index(int(s))] = q
MG = sorted({sg.main_group(s) for s in vocab})
RICH = []
for o in pool:
    c, tq = np.zeros(len(MG)), 0
    for s, q in o["subgroups"]:
        c[MG.index(sg.main_group(int(s)))] += q; tq += q
    RICH.append([*(c / max(tq, 1)), o["oc"], o["nc"], np.log(o["M"]), np.log1p(tq)])
RICH = np.array(RICH, np.float32)
print(f"{len(Y)} points; raw dim {COUNTS.shape[1]}, hand-crafted dim {RICH.shape[1]}")

243 CHO molecules


19440 points; raw dim 34, hand-crafted dim 19


In [2]:
class Head(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                               nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.f(x)

class Embed(nn.Module):
    '''Deep-Sets sum-pooled subgroup embedding (= bias-free linear map of counts) + same head.'''
    def __init__(self, n_vocab, emb=16):
        super().__init__(); self.emb = nn.Linear(n_vocab, emb, bias=False); self.head = Head(emb + 2)
    def forward(self, x): return self.head(torch.cat([self.emb(x[:, :-2]), x[:, -2:]], 1))

REPS = {"A: raw counts": (COUNTS, Head), "B: hand-crafted": (RICH, Head), "C: learned embedding": (COUNTS, Embed)}

def run(rep, train_mols, test_mols, seed, max_epochs=500, patience=30):
    F, cls = REPS[rep]
    X = np.concatenate([F[PM], CT], 1)
    tr = np.where(np.isin(PM, train_mols))[0]
    split = np.random.RandomState(0).permutation(len(tr))                  # fixed interp/val split
    interp, fit = tr[split[:int(0.2 * len(tr))]], tr[split[int(0.2 * len(tr)):]]
    nv = int(0.1 * len(fit)); val, fit = fit[:nv], fit[nv:]
    gen = np.where(np.isin(PM, test_mols))[0]
    if cls is Embed:                                                       # counts enter the embedding unscaled
        xm = np.r_[np.zeros(F.shape[1]), X[fit, -2:].mean(0)]; xs = np.r_[np.ones(F.shape[1]), X[fit, -2:].std(0)]
    else:
        xm, xs = X[fit].mean(0), X[fit].std(0); xs = np.where(xs < 1e-3, 1.0, xs)
    ym, ys = Y[fit].mean(0), Y[fit].std(0) + 1e-6
    Xn = torch.tensor(((X - xm) / xs).astype(np.float32)); Yn = torch.tensor((Y - ym) / ys)
    torch.manual_seed(seed); r = np.random.RandomState(seed)
    net = cls(F.shape[1]) if cls is Embed else cls(X.shape[1])
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(r.permutation(len(fit)))]
        for s in range(0, len(perm), 256):
            i = perm[s:s + 256]; opt.zero_grad(); L(net(Xn[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(Xn[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    with torch.no_grad(): err = np.abs(net(Xn).numpy() * ys + ym - Y)
    per_mol = pd.Series(err[gen, 1]).groupby(PM[gen]).mean()
    return {"interp_w": err[interp, 0].mean(), "interp_org": err[interp, 1].mean(),
            "gen_w": err[gen, 0].mean(), "gen_org": err[gen, 1].mean(), "epochs": ep + 1}, per_mol

In [3]:
hashv = np.array([int(hashlib.sha256(o["smiles"].encode()).hexdigest(), 16) for o in pool])
ids = np.arange(len(pool))
test_h = ids[hashv % 100 < 15]; train_h = ids[hashv % 100 >= 15]
res_h, t0 = [], time.time()
for rep in REPS:
    for seed in range(5):
        m, _ = run(rep, train_h, test_h, seed)
        res_h.append({"representation": rep, "seed": seed, **m})
    print(f"  hash split: {rep} done ({time.time()-t0:.0f}s)")
res_h = pd.DataFrame(res_h); res_h.to_csv("feature_ablation_hash_runs.csv", index=False)
fmt = lambda s: f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
tab_h = res_h.groupby("representation")[["interp_org", "gen_w", "gen_org"]].agg(fmt)
print(f"Hash split ({len(test_h)} held-out molecules), mean ± sd over 5 seeds")
display(tab_h)

  hash split: A: raw counts done (22s)


  hash split: B: hand-crafted done (48s)


  hash split: C: learned embedding done (75s)
Hash split (33 held-out molecules), mean ± sd over 5 seeds


,interp_org,gen_w,gen_org
representation,,,
A: raw counts,0.059 ± 0.010,0.052 ± 0.005,0.476 ± 0.068
B: hand-crafted,0.060 ± 0.007,0.021 ± 0.002,0.256 ± 0.025
C: learned embedding,0.062 ± 0.008,0.024 ± 0.003,0.208 ± 0.058


In [4]:
fold = hashv % 5
res_cv, per_mol = [], {rep: [] for rep in REPS}
for rep in REPS:
    for seed in range(3):
        pm_all = []
        for k in range(5):
            m, pm = run(rep, ids[fold != k], ids[fold == k], 100 * seed + k)
            res_cv.append({"representation": rep, "seed": seed, "fold": k, **m}); pm_all.append(pm)
        per_mol[rep].append(pd.concat(pm_all))
    print(f"  5-fold CV: {rep} done ({time.time()-t0:.0f}s)")
res_cv = pd.DataFrame(res_cv); res_cv.to_csv("feature_ablation_cv_runs.csv", index=False)
# per-seed molecule-averaged OOF error, then mean ± sd over seeds
oof = {rep: [s.mean() for s in per_mol[rep]] for rep in REPS}
tab_cv = pd.DataFrame({rep: {"OOF gen MAE ln γ_org": f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}"} for rep, v in oof.items()}).T
print("5-fold CV over all 243 molecules (per-molecule out-of-fold error, mean ± sd over 3 seeds)")
display(tab_cv)
mol_mean = {rep: pd.concat(per_mol[rep], axis=1).mean(1) for rep in REPS}
for a, b in [("B: hand-crafted", "A: raw counts"), ("B: hand-crafted", "C: learned embedding"), ("C: learned embedding", "A: raw counts")]:
    d = mol_mean[a] - mol_mean[b]
    print(f"{a} vs {b}: mean Δ = {d.mean():+.3f} ({100*d.mean()/mol_mean[b].mean():+.0f}%), "
          f"{a.split(':')[0]} better for {(d < 0).mean()*100:.0f}% of molecules, Wilcoxon p = {stats.wilcoxon(mol_mean[a], mol_mean[b]).pvalue:.2g}")

  5-fold CV: A: raw counts done (136s)


  5-fold CV: B: hand-crafted done (211s)


  5-fold CV: C: learned embedding done (303s)
5-fold CV over all 243 molecules (per-molecule out-of-fold error, mean ± sd over 3 seeds)


,OOF gen MAE ln γ_org
A: raw counts,0.392 ± 0.018
B: hand-crafted,0.307 ± 0.020
C: learned embedding,0.189 ± 0.007


B: hand-crafted vs A: raw counts: mean Δ = -0.086 (-22%), B better for 72% of molecules, Wilcoxon p = 9.4e-09
B: hand-crafted vs C: learned embedding: mean Δ = +0.118 (+62%), B better for 42% of molecules, Wilcoxon p = 4.1e-06
C: learned embedding vs A: raw counts: mean Δ = -0.203 (-52%), C better for 87% of molecules, Wilcoxon p = 6.3e-32


## Follow-up controls

1. **Scaling control (CHO pool).** A standardizes counts, which inflates rare subgroups, whereas C uses raw
   counts. A′ is A with *unscaled* counts.
2. **Full BIMOG pool (368 molecules, hash split, 5 seeds).** A, A′, B and C, in the same setting as
   Tables 1 and 2.
3. **MCM pool (2855 training / 482 held-out molecules, 3 seeds).** C compared with B (0.098 ± 0.001) and the
   GNN (0.078 ± 0.007) from the learning-curve notebook.

In [5]:
# generic pool builder reusing the functions above
def make_pool(entries):
    P = []
    for name, smi in entries:
        r = smiles_to_components([smi], water_as_component1=True)
        if r.removed: continue
        try:
            if not (ActivityModel(r.components).evaluate([0.9, 0.1], 298.15, basis="mass").gamma_neutral > 0).all(): continue
        except Exception: continue
        mol = Chem.MolFromSmiles(smi); mh = Chem.AddHs(mol); sym = [a.GetSymbol() for a in mh.GetAtoms()]; nc_ = max(sym.count("C"), 1)
        P.append({"name": name, "smiles": smi, "canon": Chem.MolToSmiles(mol), "subgroups": r.components[1].subgroups,
                  "M": sum(a.GetMass() for a in mh.GetAtoms()), "oc": sym.count("O") / nc_, "nc": sym.count("N") / nc_,
                  "unm": int(r.n_unmatched_atoms[0])})
    return P

def set_pool(P):
    global pool, D, PM, CT, Y, COUNTS, RICH, REPS
    pool = P; rng = np.random.RandomState(0); rows = []
    for oi, o in enumerate(pool):
        m = ActivityModel([WATER, Component(2, "o", tuple(o["subgroups"]))]); e = np.linspace(0.02, 0.98, 21)
        for w in rng.uniform(e[:-1], e[1:]):
            for T in rng.uniform(273.15, 313.15, 4):
                try:
                    lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
                    if np.isfinite(lg[:2]).all(): rows.append((oi, w, T, lg[0], lg[1]))
                except Exception: pass
    D = np.array(rows); PM = D[:, 0].astype(int)
    CT = np.column_stack([D[:, 1], (D[:, 2] - 293.15) / 20]).astype(np.float32); Y = D[:, 3:5].astype(np.float32)
    voc = sorted({int(s) for o in pool for s, q in o["subgroups"]})
    COUNTS = np.zeros((len(pool), len(voc)), np.float32)
    for i, o in enumerate(pool):
        for s, q in o["subgroups"]: COUNTS[i, voc.index(int(s))] = q
    mg = sorted({sg.main_group(s) for s in voc}); R = []
    for o in pool:
        c, tq = np.zeros(len(mg)), 0
        for s, q in o["subgroups"]: c[mg.index(sg.main_group(int(s)))] += q; tq += q
        R.append([*(c / max(tq, 1)), o["oc"], o["nc"], np.log(o["M"]), np.log1p(tq)])
    RICH = np.array(R, np.float32)
    REPS = {"A: raw counts": (COUNTS, Head), "A′: raw counts, unscaled": (COUNTS, "unscaled"),
            "B: hand-crafted": (RICH, Head), "C: learned embedding": (COUNTS, Embed)}

_run_orig = run
def run(rep, train_mols, test_mols, seed, **kw):
    F, cls = REPS[rep]
    if cls == "unscaled":                       # A′: counts unscaled, (w, T) scaled, same Head
        REPS[rep] = (F, Embed); global Embed_backup
        class _HeadNoScale(nn.Module):
            def __init__(self, n_vocab):
                super().__init__(); self.head = Head(n_vocab + 2)
            def forward(self, x): return self.head(x)
        REPS[rep] = (F, _HeadNoScale)
        _HeadNoScale.__name__ = "Embed"          # reuse the unscaled-counts branch of run()
        globals()["Embed_orig"] = Embed
        try:
            globals()["Embed"] = _HeadNoScale
            return _run_orig(rep, train_mols, test_mols, seed, **kw)
        finally:
            globals()["Embed"] = Embed_orig; REPS[rep] = (F, "unscaled")
    return _run_orig(rep, train_mols, test_mols, seed, **kw)

def hash_eval(P_mask_train, P_mask_test, reps, seeds, label):
    out = []
    for rep in reps:
        for seed in seeds:
            m, _ = run(rep, np.where(P_mask_train)[0], np.where(P_mask_test)[0], seed)
            out.append({"pool": label, "representation": rep, "seed": seed, **m})
        print(f"  {label}: {rep} done ({time.time()-t0:.0f}s)")
    return out

In [6]:
t0 = time.time(); ctrl = []
bim = json.load(open("bimog_unified.json"))
# 1) CHO pool, A′ only (A, B, C above)
set_pool(make_pool([(d["name"], d["smiles"]) for d in bim if d["#N"] == 0 and d["#Hal"] == 0]))
hv = np.array([int(hashlib.sha256(o["smiles"].encode()).hexdigest(), 16) % 100 for o in pool])
ctrl += hash_eval(hv >= 15, hv < 15, ["A′: raw counts, unscaled"], range(5), "CHO (243)")
# 2) full BIMOG
set_pool(make_pool([(d["name"], d["smiles"]) for d in bim]))
hv = np.array([int(hashlib.sha256(o["smiles"].encode()).hexdigest(), 16) % 100 for o in pool])
ctrl += hash_eval(hv >= 15, hv < 15, list(REPS), range(5), "full BIMOG (368)")

  CHO (243): A′: raw counts, unscaled done (27s)


  full BIMOG (368): A: raw counts done (64s)


  full BIMOG (368): A′: raw counts, unscaled done (107s)


  full BIMOG (368): B: hand-crafted done (148s)


  full BIMOG (368): C: learned embedding done (201s)


In [7]:
# 3) MCM pool: C (learned embedding), 3 seeds; same filter and de-duplication as the learning-curve notebook
bcanon = {o["canon"] for o in make_pool([(d["name"], d["smiles"]) for d in bim])}
mraw = pd.read_csv("mcm_v331_species.tsv", sep="\t", comment="*"); ent = []
for _, r in mraw.iterrows():
    mol = Chem.MolFromSmiles(str(r.Smiles))
    if (mol is None or str(r.Excited).lower() == "true" or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms())
            or not {a.GetSymbol() for a in mol.GetAtoms()} <= {"C", "H", "O", "N"} or mol.GetNumHeavyAtoms() < 2
            or "C" not in {a.GetSymbol() for a in mol.GetAtoms()}): continue
    ent.append((r.Name, str(r.Smiles)))
P = [o for o in make_pool(ent) if o["unm"] == 0 and o["canon"] not in bcanon]
set_pool(P)
hv = np.array([int(hashlib.sha256(o["smiles"].encode()).hexdigest(), 16) % 100 for o in pool])
print(f"MCM pool {len(pool)} (train {(hv >= 15).sum()}, held-out {(hv < 15).sum()})")
ctrl += hash_eval(hv >= 15, hv < 15, ["C: learned embedding"], range(3), "MCM (3337)")
ctrl = pd.DataFrame(ctrl); ctrl.to_csv("feature_ablation_controls.csv", index=False)
fmt = lambda s: f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
display(ctrl.groupby(["pool", "representation"], sort=False)[["interp_org", "gen_w", "gen_org"]].agg(fmt))

MCM pool 3337 (train 2855, held-out 482)


  MCM (3337): C: learned embedding done (413s)


interp_org          gen_w  \
pool             representation                                           
CHO (243)        A′: raw counts, unscaled  0.061 ± 0.010  0.031 ± 0.003   
full BIMOG (368) A: raw counts             0.056 ± 0.005  0.033 ± 0.002   
                 A′: raw counts, unscaled  0.057 ± 0.009  0.024 ± 0.001   
                 B: hand-crafted           0.063 ± 0.010  0.024 ± 0.003   
                 C: learned embedding      0.054 ± 0.006  0.016 ± 0.001   
MCM (3337)       C: learned embedding      0.026 ± 0.001  0.007 ± 0.000   

                                                 gen_org  
pool             representation                           
CHO (243)        A′: raw counts, unscaled  0.259 ± 0.004  
full BIMOG (368) A: raw counts             0.422 ± 0.064  
                 A′: raw counts, unscaled  0.257 ± 0.016  
                 B: hand-crafted           0.307 ± 0.022  
                 C: learned embedding      0.161 ± 0.008  
MCM (3337)       C: learned embedding      0.031 ± 0.002